# Differential regulation: T-test for two groups

This API example shows the functionality in the [`acore.differential_regulation`](acore.differential_regulation) module:

- ANOVA (for two groups) using `run_anova` is equivalent to a t-test between two groups
using `run_ttest`. It's best to use `run_anova` for two groups, as it will
automatically call `run_ttest` and return the same results.

Then we can do the same for examples with three
and more groups, where a omnibus analysis across groups
is combined with posthoc anlysis for separate groups.

The functions are the same for both cases. The `group1` and
`group2` columns give the posthoc comparison.


In [ ]:
%pip install acore vuecore

In [ ]:
import dsp_pandas
import pandas as pd
import vuecore.interactive

import acore.differential_regulation as ad

dsp_pandas.format.set_pandas_options(
    max_columns=9,
    max_colwidth=20,
)

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/Multiomics-Analytics-Group/acore/"
    "main/example_data/alzheimer_proteomics/"
)
# data is already preprocessed: log2, filtered
fname: str = "alzheimer_example_omics_and_clinic.csv"  # combined omics and meta data
covariates: list[str] = ["age", "male"]
group: str = "AD"
subject_col: str = "Sample ID"
drop_cols: list[str] = ["collection_site"]
factor_and_covars: list[str] = [group, *covariates]

Alternatively you can use the metabolomics dataset MTBLS13311,
which is also preprocessed and combined.

In [ ]:
# BASE = (
#     "https://raw.githubusercontent.com/Multiomics-Analytics-Group/acore/"
#     "main/example_data/MTBLS13311/"
# )
# fname: str = "MTBLS13411_omics_and_meta.csv"  # combined omics and meta data
# covariates: list[str] = []
# group: str = "Factor Value[Strain type]"
# drop_cols: list[str] = ["group"]
# subject_col: str | int = 0
# factor_and_covars: list[str] = [group, *covariates]

## ANOVA analysis for two groups
Use combined dataset for ANOVA analysis.
Drop unnecessary columns, if there are any specified in `drop_cols`.

In [ ]:
omics_and_meta = (
    pd.read_csv(f"{BASE}/{fname}", index_col=subject_col)
    .convert_dtypes()
    .dropna(subset=factor_and_covars)
    .drop(columns=drop_cols)
)
omics_and_meta

Check data types of the columns. Metadata can be numeric, but also strings.

In [ ]:
omics_and_meta.dtypes.value_counts()

In [ ]:
omics_and_meta[[group, *covariates]]

## ANOVA analysis for two groups
- is not controlling for covariates
- is equivalent to a t-test between two groups

In [ ]:
if not isinstance(subject_col, str):
    subject_col = omics_and_meta.index.name or "index"
    omics_and_meta.rename_axis(subject_col, axis=0, inplace=True)
anova = (
    ad.run_anova(
        omics_and_meta,
        subject=None,
        drop_cols=covariates,
        group=group,
    )
    .set_index("identifier")
    .sort_values(by="padj")
)
anova

See summary statistics for numeric values in ANOVA results, such as for
- pvalue (unadjusted)
- padj (adjusted p-value, FDR corrected)
- fold change (FC) of the mean of group1 vs group2 (group1/group2)

In [ ]:
anova.describe().T

## Inspect ANOVA results
- summary statistics for each group
- test results
- t-test statistic, FDR correction method, etc.

### View averages per protein group

In [ ]:
view = anova.filter(regex="group")
viewed_cols = view.columns.to_list()
view

### Test results

In [ ]:
regex_filter = "pval|padj|reject|stat|FC"
view = anova.filter(regex=regex_filter)
viewed_cols.extend(view.columns)
view

### Other information
on test statistic and FDR correction method.

In [ ]:
anova.drop(columns=viewed_cols)

## T-test for two groups
`run_anova` calls `run_ttest` for two groups. You can also call
[`run_ttest`](acore.differential_regulation.run_ttest) directly, which
requires the two conditions to compare. The result adheres to the
`TtestSchema` and contains one row per feature.

- use `paired=True` and `subject` for paired samples
- use `non_par=True` for a Mann-Whitney U test instead of a t-test

In [ ]:
condition1, condition2 = omics_and_meta[group].unique()[:2]
ttest = (
    ad.run_ttest(
        omics_and_meta,
        condition1=condition1,
        condition2=condition2,
        subject=None,
        drop_cols=covariates,
        group=group,
    )
    .set_index("identifier")
    .sort_values(by="padj")
)
ttest

The t-test results are the same as for the ANOVA with two groups.

In [ ]:
pd.testing.assert_series_equal(
    ttest["pvalue"], anova["pvalue"].loc[ttest.index], check_names=False
)

In [ ]:
ttest.filter(regex="pvalue|padj|FC|rejected").head()

## View Volcano plot for the ANOVA results
- p-value (unadjusted) using the -log10 transformation on the y-axis, using
   `-log10 pvalue`
- fold change (FC) in the log2 space on the x-axis (2 means a fold
  change of 4, 1 a doubeling and -1 means a fold change of 0.5 and -2 of 0.25),
  using `log2FC`
- significant features are colored in red using the `reject` column,
  which is True for significant features and False for non-significant features, and
  is calculated based on the `padj` column and a significance threshold (default 0.05)

In [ ]:
vuecore.interactive.get_differential_regulation_plot(
    data=anova.reset_index(),
)

Done.